# 02 (Kaggle판). 학습 / QAT

Colab 02 노트북과 **학습 코드는 같고**, 저장 위치와 이어하기 방법만 Kaggle에 맞췄어요.

- Kaggle 무료 GPU: **주당 약 30시간**, 한 번 실행은 **최대 12시간**
- **Save & Run All(커밋)** 으로 돌리면 **브라우저를 꺼도** 서버에서 끝까지 돌아가요 (Colab처럼 화면을 켜 둘 필요 없음)
- 12시간을 넘기면 결과가 날아갈 수 있어서, `TIME_LIMIT_H`(기본 11시간)가 지나면 **바퀴가 끝난 직후 저장하고 스스로 멈춰요.** 다음 실행에서 이어서 해요.

## 처음 한 번만 준비

1. kaggle.com 가입 → 프로필 설정에서 **전화번호 인증** (GPU·인터넷을 켜려면 필요)
2. **선생 가중치 올리기** (증류할 때만)
   - Colab Drive의 `runs/yolov8s_960_e60/weights/best.pt` 를 내려받아요
   - 파일 이름을 **`yolov8s_960_e60.pt`** 로 바꿔요 (실험 이름 + `.pt`)
   - Kaggle → Datasets → **New Dataset** → 파일 올리기 (데이터셋 이름은 아무거나, 예: `kitti-weights`)
   - QAT만 할 때는 출발 모델(`BASE`)의 `best.pt` 를 같은 방법으로 `BASE이름.pt` 로 올려요
3. 이 노트북 가져오기: Kaggle → Code → **New Notebook** → File → **Import Notebook** → 이 파일 올리기
4. 오른쪽 **Settings**: Accelerator = **GPU T4 x2**, Internet = **On**
5. 오른쪽 **Add Input** → 2번에서 만든 데이터셋 추가

## 실행

1. 아래 **1번 칸 설정**만 바꿔요 (Colab 02의 5번 칸과 같은 값)
2. 오른쪽 위 **Save Version** → **Save & Run All (Commit)** → 이제 브라우저를 꺼도 돼요
3. 진행 상황: 노트북 페이지 → 버전 → **Logs**

## 이어서 하기 (11시간에 멈췄을 때)

1. 같은 노트북 편집 화면 → **Add Input** → **Your Work / Notebooks** → 이 노트북의 **방금 끝난 버전 Output** 추가
2. 설정 그대로 다시 **Save & Run All** → 3번 칸이 이전 결과(`runs/`)를 가져와서 이어서 학습해요

## Colab에서 하다 멈춘 학습을 Kaggle에서 이어서 하기

1. Colab Drive의 `runs/실험이름/weights/last.pt` 를 내려받아 **`실험이름_last.pt`** 로 이름을 바꿔요
   (예: `yolo11n_1152_e90_kd_cw_ped_last.pt`)
2. 선생 가중치와 **같은 데이터셋**에 올려요 (데이터셋 페이지 → New Version → 파일 추가)
3. 1번 칸을 **Colab 때와 같은 설정**으로 두고 Save & Run All → `이어서 학습` 이 나오면 성공
   - 체크포인트 안의 Colab 경로(`/content/...`)는 Kaggle 경로로 자동으로 바꿔요
   - ⚠️ **QAT 는 이어하기가 안 돼요** (Ultralytics 버그). QAT 는 `_last.pt` 를 올리지 말고 처음부터 다시 해요.
     10바퀴라 한 번 실행(11시간) 안에 끝나요.

## 결과를 Colab 03으로 채점하기

1. 버전의 **Output** 탭에서 `runs/실험이름/weights/best.pt` 내려받기
2. Colab Drive의 `runs/실험이름/weights/` 폴더에 올리기 (폴더가 없으면 만들기)
3. Colab 03 노트북(CPU)에서 `NAME = '실험이름'` 으로 평소처럼 채점

## 1. 설정 ⭐ 여기만 바꿔요

In [ ]:
TASK = 'train'      # 'train' = 학습, 'qat' = QAT (Colab 02의 8번 칸 역할)

# --- TASK = 'train' 일 때 (Colab 02의 5번 칸과 같은 뜻) ---
MODEL = 'yolo11n'   # yolov8n / yolov8s / yolo11n / yolo11m
IMGSZ = 1280
EPOCHS = 90
TEACHER = 'yolo11n_1280_e90_kd_cw_ped3_s0.9'   # 증류 선생 실험 이름 (Add Input 에 있는 가중치). 안 쓰면 None (따옴표 없이)
KD_TAG = '2'        # 선생을 바꾸면 숫자도 바꿔요. 실험 이름이 _kd2 가 돼서 예전 실험과 안 겹쳐요 ('' = 예전 이름)
CLS_PW = 0.5        # 클래스 가중치: 0 = 끔, 0.5, 1.0
FULL = False        # True = dev 도 학습에 넣기 (설정을 다 정한 뒤 마지막 재학습 때만. dev 점수는 믿을 수 없어요)
PED = True          # True = DontCare 가리기 + 사람 사진 여러 번
PED_DUP = 3         # 사람 사진을 한 바퀴에 몇 번 볼지 (예전 PED 실험 = 2)
PED_HARD = True     # True = '어려운 사람'이 있는 사진만 PED_DUP 번, 나머지 사람 사진은 1번
PED_CROP = True     # True = 어려운 사람 주변을 2배 확대한 사진을 1장씩 추가
SCALE = 0.9         # 크기 증강: 사진을 ±몇 배까지 키우고 줄일지 (기본 0.5)

# --- TASK = 'qat' 일 때 ---
BASE = 'yolo11n_1280_e90_kd2_cw_ped3hc_s0.9'   # QAT 출발 실험 이름
QAT_EPOCHS = 10
QAT_SCALE = None    # None = 출발 모델과 같은 크기 증강. 숫자(예: 0.5)를 넣으면 그 값으로 QAT 하고 이름에 _qs0.5 가 붙어요

TIME_LIMIT_H = 11   # 이 시간이 지나면 바퀴가 끝난 직후 저장하고 멈춤 (Kaggle 한 번 실행 최대 12시간)

import re, time
START = time.time()   # 준비 시간까지 포함해서 잼

assert TASK in ('train', 'qat')
cw = '' if not CLS_PW else '_cw' if CLS_PW == 0.5 else f'_cw{CLS_PW}'
ped_tag = f'{PED_DUP}' + ('h' if PED_HARD else '') + ('c' if PED_CROP else '')
ped = '' if not PED else '_ped' if ped_tag == '2' else f'_ped{ped_tag}'
NAME = (f'{MODEL}_{IMGSZ}_e{EPOCHS}' + (f'_kd{KD_TAG}' if TEACHER else '') + cw + ('_full' if FULL else '') + ped
        + ('' if SCALE == 0.5 else f'_s{SCALE}'))

def data_tag(name):   # 실험 이름에서 PED 데이터 종류 읽기. 예: '2', '3h', '3hc', '3hcf' (f = dev 포함)
    m = re.search(r'_ped(\d*h?c?)', name)
    return ((m.group(1) or '2') + ('f' if '_full' in name else '')) if m else None

def dc_data(tag):   # PED 데이터 위치 (종류마다 따로)
    return '/tmp/kitti_yolo_dc/kitti_dc.yaml' if tag == '2' else f'/tmp/kitti_yolo_dc{tag}/kitti_dc.yaml'

def ped_args(tag):   # make_ped_data.py 에 넘길 옵션
    n = re.match(r'\d+', tag).group()
    args = ['--dup', '1', '--hard-dup', n] if 'h' in tag else ['--dup', n]
    return args + (['--crop', '1'] if 'c' in tag else []) + (['--full'] if 'f' in tag else [])

if PED:
    DATA = dc_data(data_tag(NAME))
elif FULL:
    DATA = '/tmp/kitti_yolo/kitti_full.yaml'
else:
    DATA = '/tmp/kitti_yolo/kitti.yaml'
if MODEL.endswith('m'):
    BATCH = 4
elif TEACHER and IMGSZ >= 1280:
    BATCH = 8
elif IMGSZ >= 1152 or MODEL.endswith('s') or TEACHER:
    BATCH = 16
else:
    BATCH = 32
QAT_NAME = f'{BASE}_qat{QAT_EPOCHS}' + ('' if QAT_SCALE is None else f'_qs{QAT_SCALE}')
if TASK == 'train':
    print('실험 이름:', NAME, '| batch', BATCH, '| 데이터', DATA)
else:
    print('QAT 출발:', BASE, '→', QAT_NAME)

## 2. 데이터 + 도구 준비 (약 20~30분)

데이터는 `/tmp` 에 둬요. `/kaggle/working` 은 결과 저장용(20GB 제한)이라 데이터를 넣으면 넘쳐요.

In [ ]:
%%bash
set -e
git clone -q https://github.com/gyu-young-04/embeded_ondeviceAI.git /tmp/embeded_ondeviceAI 2>/dev/null || git -C /tmp/embeded_ondeviceAI pull -q
mkdir -p /tmp/kitti && cd /tmp/kitti
for f in data_object_label_2.zip data_object_image_2.zip; do
  if [ ! -f "done_$f" ]; then
    wget -c -q https://s3.eu-central-1.amazonaws.com/avg-kitti/$f
    unzip -q -n $f 'training/*'
    rm -f $f && touch "done_$f"   # 디스크 아끼려고 압축 파일은 지움
  fi
done
[ "$(ls training/image_2 | wc -l)" = 7481 ] || { echo '사진이 7481장이 아니에요. /tmp/kitti 를 지우고 다시 실행하세요.'; exit 1; }
cd /tmp/embeded_ondeviceAI
[ -f /tmp/kitti_yolo/kitti.yaml ] || python scripts/prepare_kitti.py --kitti /tmp/kitti --out /tmp/kitti_yolo
[ -f /tmp/kitti_yolo/kitti_full.yaml ] || sed 's#^train: train.txt#train: [train.txt, dev.txt]#' /tmp/kitti_yolo/kitti.yaml > /tmp/kitti_yolo/kitti_full.yaml
wc -l /tmp/kitti_yolo/train.txt /tmp/kitti_yolo/dev.txt
df -h /tmp /kaggle/working | cat

In [ ]:
!pip -q install ultralytics==8.4.189 nvidia-modelopt==0.48.0   # Colab 과 같은 버전 (QAT 파일 호환)
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__,
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')
assert torch.cuda.is_available(), 'GPU 가 꺼져 있어요. 오른쪽 Settings → Accelerator 를 GPU 로 바꾸세요.'

# PED 데이터 만들기 (학습이든 QAT 든 필요한 것만, 약 5~10분)
import subprocess
from pathlib import Path
tag = data_tag(NAME if TASK == 'train' else BASE)
if tag:
    if not Path(dc_data(tag)).exists():
        subprocess.run(['python', 'scripts/make_ped_data.py', '--kitti', '/tmp/kitti', '--src', '/tmp/kitti_yolo',
                        '--out', str(Path(dc_data(tag)).parent), *ped_args(tag)], cwd='/tmp/embeded_ondeviceAI', check=True)
    print('PED 데이터:', dc_data(tag))

## 3. 이전 결과 · 가중치 가져오기

- **Add Input** 에 이전 버전 Output 이 있으면 그 안의 `runs/` 를 가져와서 이어서 해요.
- 선생·QAT 출발 가중치는 Add Input 에서 `실험이름.pt` (또는 `실험이름/weights/best.pt`) 를 찾아요.

In [ ]:
import shutil
from pathlib import Path

RUNS = Path('/kaggle/working/runs')
RUNS.mkdir(parents=True, exist_ok=True)
INPUT = Path('/kaggle/input')

# 이전 Kaggle 버전 Output 의 runs/ 를 가져오기 (이미 있으면 건너뜀)
for prev in INPUT.rglob('runs'):
    if not prev.is_dir():
        continue
    for run_dir in prev.iterdir():
        if run_dir.is_dir() and not (RUNS / run_dir.name).exists():
            shutil.copytree(run_dir, RUNS / run_dir.name)
            print('이전 결과 가져옴:', run_dir)

# Colab 에서 하다 멈춘 학습: last.pt 를 '실험이름_last.pt' 로 바꿔 올리면 여기로 가져와요
for ck in INPUT.rglob('*_last.pt'):
    name = ck.name[:-len('_last.pt')]
    dst = RUNS / name / 'weights' / 'last.pt'
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(ck, dst)
        print('Colab 에서 하던 학습 가져옴:', ck, '→', dst)

def find_weights(name):
    hits = [RUNS / name / 'weights' / 'best.pt'] + sorted(INPUT.rglob(f'{name}/weights/best.pt')) + sorted(INPUT.rglob(f'{name}.pt'))
    hits = [h for h in hits if h.exists()]
    assert hits, f'{name} 가중치를 못 찾았어요. {name}.pt 로 이름을 바꿔 데이터셋에 올리고 Add Input 에 추가하세요.'
    print(name, '가중치:', hits[0])
    return hits[0]

for run_dir in sorted(RUNS.iterdir()):
    print('runs/', run_dir.name)

## 4. 학습 / QAT

In [ ]:
import torch
from ultralytics import YOLO

class TimeUp(Exception):
    pass

def stop_if_time(trainer):
    # last.pt 를 저장한 직후에 불려요. 여기서 멈춰도 last.pt 로 이어서 할 수 있어요
    if (time.time() - START) / 3600 > TIME_LIMIT_H:
        raise TimeUp

def finished(ckpt):
    # Ultralytics 는 학습이 끝나면 체크포인트의 epoch 를 -1 로 바꿔 둔다
    return torch.load(ckpt, map_location='cpu', weights_only=False).get('epoch', 0) == -1

def run(name, start_weights, resume_kw, can_resume=True, **train_kw):
    last = RUNS / name / 'weights' / 'last.pt'
    if last.exists() and finished(last):
        print('이미 끝난 실험이에요:', name)
        return
    if last.exists() and not can_resume:
        print('⚠️ QAT 는 이어하기가 안 돼서(Ultralytics 버그) 처음부터 다시 해요:', name)
        shutil.rmtree(RUNS / name)
    model = YOLO(str(last if last.exists() else start_weights))
    model.add_callback('on_model_save', stop_if_time)
    try:
        if last.exists():
            print('이어서 학습:', last)
            # 체크포인트 안의 경로가 Colab(/content/...)이어도 Kaggle 경로로 바꿔서 이어가요
            model.train(resume=True, data=train_kw['data'], save_dir=str(RUNS / name), device=0, cache='ram', **resume_kw)
        else:
            model.train(project=str(RUNS), name=name, exist_ok=True, seed=0, deterministic=True,
                        cache='ram', device=0, **train_kw)
        print('✅ 끝났어요:', name)
    except TimeUp:
        print(f'⏸ {TIME_LIMIT_H}시간이 지나서 저장하고 멈췄어요. 이 버전 Output 을 Add Input 에 넣고 다시 Save & Run All 하세요.')

if TASK == 'train':
    extra = {'distill_model': str(find_weights(TEACHER))} if TEACHER else {}
    run(NAME, f'{MODEL}.pt', {'batch': BATCH, 'scale': SCALE, **extra},
        data=DATA, imgsz=IMGSZ, epochs=EPOCHS, patience=20, batch=BATCH, cls_pw=CLS_PW, scale=SCALE, **extra)
else:
    imgsz = int(BASE.split('_')[1])
    tag = data_tag(BASE)
    ms = re.search(r'_s([0-9.]+)$', BASE)
    if tag:
        QDATA = dc_data(tag)
    elif '_full' in BASE:
        QDATA = '/tmp/kitti_yolo/kitti_full.yaml'
    else:
        QDATA = '/tmp/kitti_yolo/kitti.yaml'
    QCLS_PW = 1.0 if '_cw1.0' in BASE else 0.5 if '_cw' in BASE else 0
    QSCALE = QAT_SCALE if QAT_SCALE is not None else float(ms.group(1)) if ms else 0.5
    QBATCH = 8 if imgsz >= 1280 else 16
    print('QAT | 데이터', QDATA, '| cls_pw', QCLS_PW, '| scale', QSCALE, '| batch', QBATCH)
    run(QAT_NAME, find_weights(BASE), {}, can_resume=False,
        data=QDATA, imgsz=imgsz, epochs=QAT_EPOCHS, quantize=8, optimizer='SGD', lr0=0.001,
        warmup_epochs=0, close_mosaic=QAT_EPOCHS, cls_pw=QCLS_PW, scale=QSCALE, batch=QBATCH)

## 5. 결과 확인 (Output 탭에서 내려받을 파일)

In [ ]:
import pandas as pd
rows = []
for run_dir in sorted(RUNS.iterdir()):
    csv = run_dir / 'results.csv'
    if not csv.exists():
        continue
    df = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    last = run_dir / 'weights' / 'last.pt'
    rows.append({'실험': run_dir.name, '학습 바퀴': len(df),
                 'dev mAP50-95 (최고)': round(df['metrics/mAP50-95(B)'].max(), 3),
                 '상태': '완료' if last.exists() and finished(last) else '진행 중 (이어서 하기)'})
print(pd.DataFrame(rows).to_string(index=False))
print('\n내려받을 파일:')
for p in sorted(RUNS.rglob('best.pt')):
    print(' ', p)